# Track Separation — 음원 트랙 분리

음원 파일 하나를 넣으면 **세 가지 트랙**으로 분리합니다:

| 출력 트랙 | 내용 |
|-----------|------|
| **Vocals** | 보컬 (메인 + 코러스) |
| **Melody** | 멜로디 악기 (기타, 피아노, 신스, 베이스 등) |
| **Drums** | 비트 악기 (드럼, 퍼커션) |

내부적으로 Meta의 **Demucs v4 (htdemucs)** 모델을 사용합니다. Demucs는 vocals / drums / bass / other 4트랙으로 분리하는데, 여기서 bass + other를 합쳐 "멜로디 악기" 트랙으로 만듭니다.

## 사용법

1. 아래 셀들을 순서대로 실행
2. 마지막 셀에서 `TRACK_PATH`를 분리하고 싶은 파일 경로로 변경
3. 결과는 원본 파일 옆에 `{파일명}_stems/` 폴더로 저장됩니다

## 0. 의존성 설치

처음 한 번만 실행하세요. Demucs는 PyTorch 기반이므로 torch가 함께 설치됩니다.

In [ ]:
# %pip install -q demucs soundfile numpy matplotlib IPython

In [ ]:
import os
import torch
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {'MPS (Apple Silicon)' if torch.backends.mps.is_available() else 'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 1. 분리 함수

Demucs `htdemucs` 모델로 4트랙(vocals, drums, bass, other)을 분리한 뒤,
bass + other를 합쳐 **melody** 트랙으로 만듭니다.

- Apple Silicon Mac에서는 MPS 가속을 자동으로 사용합니다.
- 첫 실행 시 모델 가중치를 다운로드합니다 (~80MB).

In [ ]:
from demucs.pretrained import get_model
from demucs.apply import apply_model


def load_audio_for_demucs(path: str, model_samplerate: int) -> torch.Tensor:
    """soundfile로 오디오를 로드하고 모델 샘플레이트에 맞춰 리샘플링한다.
    
    Returns: (channels, samples) 텐서
    """
    import librosa

    # soundfile 경유로 로드 (torchaudio/torchcodec 의존 회피)
    data, sr = sf.read(path, dtype="float32")
    # (samples,) or (samples, channels) → (channels, samples)
    if data.ndim == 1:
        data = np.stack([data, data])
    else:
        data = data.T
    # 모노면 스테레오로 확장
    if data.shape[0] == 1:
        data = np.concatenate([data, data], axis=0)

    wav = torch.from_numpy(data)

    # 리샘플링
    if sr != model_samplerate:
        wav_resampled = torch.from_numpy(
            librosa.resample(wav.numpy(), orig_sr=sr, target_sr=model_samplerate)
        )
        wav = wav_resampled

    return wav


def separate_tracks(path: str, model_name: str = "htdemucs") -> dict:
    """음원을 보컬 / 멜로디 악기 / 드럼 세 트랙으로 분리한다.
    
    Args:
        path: 음원 파일 경로 (.wav, .flac, .mp3)
        model_name: Demucs 모델 이름 (기본: htdemucs)
    
    Returns:
        dict with keys: vocals, melody, drums, sample_rate
        각 값은 (channels, samples) numpy array
    """
    # 모델 로드
    model = get_model(model_name)
    model.eval()
    
    # 디바이스 선택
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    model.to(device)
    
    print(f"Model: {model_name} on {device}")
    print(f"Sources: {model.sources}")
    
    # 오디오 로드
    wav = load_audio_for_demucs(path, model.samplerate)
    print(f"Audio loaded: {wav.shape[1] / model.samplerate:.1f}s, {model.samplerate}Hz")
    
    # 분리 실행 — (batch, sources, channels, samples)
    ref = wav.mean(0)
    wav_input = (wav - ref.mean()) / ref.std()
    
    with torch.no_grad():
        sources = apply_model(
            model,
            wav_input[None].to(device),
            shifts=1,
            overlap=0.25,
        )[0]
    
    # 역정규화
    sources = sources * ref.std() + ref.mean()
    sources = sources.cpu().numpy()
    
    # 소스 이름 → 인덱스 매핑
    src_idx = {name: i for i, name in enumerate(model.sources)}
    
    vocals = sources[src_idx["vocals"]]
    drums = sources[src_idx["drums"]]
    bass = sources[src_idx["bass"]]
    other = sources[src_idx["other"]]
    
    # bass + other = melody (멜로디 악기 트랙)
    melody = bass + other
    
    print("Separation complete!")
    print(f"  Vocals: {vocals.shape}")
    print(f"  Melody (bass+other): {melody.shape}")
    print(f"  Drums:  {drums.shape}")
    
    return {
        "vocals": vocals,
        "melody": melody,
        "drums": drums,
        "sample_rate": model.samplerate,
    }

## 2. 저장 함수

분리된 트랙을 wav 파일로 저장합니다.

In [ ]:
def save_stems(result: dict, input_path: str, out_dir: str = None) -> str:
    """분리된 트랙을 wav 파일로 저장한다.
    
    출력 디렉토리: {원본파일명}_stems/
    """
    stem = os.path.splitext(os.path.basename(input_path))[0]
    if out_dir is None:
        out_dir = os.path.join(os.path.dirname(input_path), f"{stem}_stems")
    os.makedirs(out_dir, exist_ok=True)
    
    sr = result["sample_rate"]
    
    for name in ["vocals", "melody", "drums"]:
        out_path = os.path.join(out_dir, f"{stem}_{name}.wav")
        # (channels, samples) → (samples, channels) for soundfile
        data = result[name].T
        sf.write(out_path, data, sr)
        duration = data.shape[0] / sr
        print(f"  {name:>8}: {out_path} ({duration:.1f}s)")
    
    print(f"\nSaved to {out_dir}/")
    return out_dir

## 3. 시각화 + 시청

각 트랙의 파형을 비교하고, 노트북에서 바로 들어볼 수 있습니다.

In [ ]:
def plot_stems(result: dict, input_path: str):
    """세 트랙의 파형을 나란히 시각화한다."""
    sr = result["sample_rate"]
    stem_name = os.path.splitext(os.path.basename(input_path))[0]
    
    tracks = [
        ("Vocals (보컬)", result["vocals"]),
        ("Melody (멜로디 악기)", result["melody"]),
        ("Drums (비트)", result["drums"]),
    ]
    
    fig, axes = plt.subplots(len(tracks), 1, figsize=(14, 6), sharex=True)
    colors = ["#e74c3c", "#3498db", "#2ecc71"]
    
    for ax, (name, data), color in zip(axes, tracks, colors):
        # 모노로 다운믹스해서 시각화
        mono = data.mean(axis=0)
        t = np.arange(len(mono)) / sr
        ax.plot(t, mono, lw=0.4, color=color, alpha=0.8)
        ax.fill_between(t, mono, alpha=0.15, color=color)
        ax.set_ylabel(name, fontsize=10)
        ax.set_xlim(0, t[-1])
        ax.grid(axis="x", alpha=0.3)
    
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Track Separation: {stem_name}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def listen_stems(result: dict):
    """각 트랙을 노트북 플레이어로 재생할 수 있게 표시한다."""
    sr = result["sample_rate"]
    
    labels = {
        "vocals": "Vocals (보컬)",
        "melody": "Melody (멜로디 악기)",
        "drums": "Drums (비트)",
    }
    
    for key, label in labels.items():
        display(HTML(f"<h4>{label}</h4>"))
        display(Audio(data=result[key], rate=sr))

## 4. 실행

아래 `TRACK_PATH`를 분리하고 싶은 파일로 바꾸고 실행하세요.

CPU에서는 곡 길이의 약 1~2배 시간이 걸립니다 (3분 곡 → 3~6분).
MPS/CUDA가 있으면 훨씬 빠릅니다.

In [ ]:
TRACK_PATH = "samples/your_audio.flac"  # samples/에 파일 넣고 경로 수정

# 트랙 분리
result = separate_tracks(TRACK_PATH)

# wav 파일로 저장
save_stems(result, TRACK_PATH)

In [ ]:
# 파형 시각화
plot_stems(result, TRACK_PATH)

In [ ]:
# 각 트랙 들어보기
listen_stems(result)